# K-center problem

The k-center problem is a facility location problem. A facility location problem is an optimization problem where the goal is to decide where to place some faciltiies (hospitals, warehouses, schools etc.) such that they serve a set of clients as efficiently as possible. The k-center problem solves the problem of choosing facilties such that the farthest served client is not too far away. For example a hospital serving two cities is better placed in the middle of those cities then in either one. Formally, the problem is defined like this: 

In the k-center problem, we are given a finite metric space $(X,d)$ where $d: P \times P \to \mathbb{R}_{\geq 0} $, and a set of points $P \subseteq X$, and are tasked to find a set of $k \in \mathbb{N}$ points(also called centers) $C \subseteq P$ such that the maximum distance from every point $p \in P$ to it's closest center is minimzed. That is, find $C$ so that $$cost(P,C)=\max_{p\in P}\min_{c \in C}d(p,c)$$ is minimized.

In [2]:
import numpy as np
from itertools import combinations

In [3]:
def manhattan_distance(x: list[int],y:list[int]):
    if len(x) != len(y):
        raise ValueError("Points has to be of same dimension")
    distance = sum(abs(i-j) for i,j in zip(x,y))
    return distance

def euclidean_distance(x: list[int], y:list[int]):
    if len(x) != len(y):
        raise ValueError("Points has to be of same dimension")
    if len(x) == 1:
        return abs(x[0]-y[0])
    distance = np.sqrt(sum(np.pow(i-j,2) for i,j in zip(x,y)))
    return distance

In [4]:
def build_dataset(n, min_val=1, max_val=200, dim=1):
    dataset = np.unique(np.random.randint(min_val, max_val, (n, dim)), axis=0)
    while len(dataset) < n:
        new_point = np.random.randint(min_val, max_val, (1, dim))
        dataset = np.unique(np.vstack([dataset, new_point]), axis=0)
    return dataset

dataset = build_dataset(n=10,dim=2)
print(f"dataset: {dataset}")


dataset: [[  3  65]
 [  4 147]
 [ 67 167]
 [ 73  38]
 [ 99   5]
 [114 187]
 [131 126]
 [133 185]
 [140  58]
 [147  20]]


# K-center brute force algorithm

In [5]:
k = 3
candidate_center_sets = list(combinations(dataset,k))
print(f"number of possible different clusterings with {k} centers, in a dataset of {len(dataset)} points: {len(candidate_center_sets)}")
#print(f"candidate center sets: {candidate_center_sets}")

def k_center_brute(dataset, candidate_center_sets, d, k):
    """
    Returns:
        (optimal_centers, optimal_cost): the best set of centers and its cost.
    """
    clustering_costs:dict[int,int] = {} # {clustering_index: cost}

    for i, center_set in enumerate(candidate_center_sets):

        distances_to_nearest_center = []
        
        for point in dataset:
            distance_to_nearest_center = min(
                d(point, center)
                for center in center_set
            )

            distances_to_nearest_center.append(distance_to_nearest_center)
        clustering_costs[i] = max(distances_to_nearest_center)
    
    optimal_center_set_idx = min(clustering_costs, key=lambda k: clustering_costs[k])
    optimal_centers = candidate_center_sets[optimal_center_set_idx]
    optimal_cost = clustering_costs[optimal_center_set_idx]
    return optimal_centers, optimal_cost

def extract_clustering(dataset: np.ndarray, centers, d):
    """
    Returns:
        A dict mapping each center's index to the list of points assigned to it (centers included).
    """
    clusters = {i: [] for i in range(len(centers))}
    for p in dataset:
        nearest = min(range(len(centers)), key=lambda i: d(p, centers[i])) #argmin on range(len(centers)) compared on d(p,c) / argmin pattern
        clusters[nearest].append(p)
    return clusters


optimal_centers, optimal_cost = k_center_brute(dataset=dataset, candidate_center_sets=candidate_center_sets, d=euclidean_distance, k = k)
print(f"optimal_center_set: {optimal_centers} with cost: {optimal_cost}")

clusters = extract_clustering(dataset=dataset, centers=optimal_centers, d=euclidean_distance)
print(f"clusters: {clusters}")


number of possible different clusterings with 3 centers, in a dataset of 10 points: 120
optimal_center_set: (array([ 3, 65]), array([ 67, 167]), array([140,  58])) with cost: 69.92138442565336
clusters: {0: [array([ 3, 65])], 1: [array([  4, 147]), array([ 67, 167]), array([114, 187]), array([133, 185])], 2: [array([73, 38]), array([99,  5]), array([131, 126]), array([140,  58]), array([147,  20])]}


### Vectorized version of the brute-force algo

In [8]:
from scipy.spatial.distance import cdist

def k_center_brute_vectorized(dataset, k, metric="euclidean"):  # "cityblock" = manhattan
    D = cdist(dataset, dataset, metric=metric)        # type: ignore  # (n, n) pairwise distances
    best_cost, best_candidate_center_set = np.inf, None
    for candidate_center_set in combinations(range(len(dataset)), k):
        cost = D[:, list(candidate_center_set)].min(axis=1).max()    # nearest-center per point, then worst point
        if cost < best_cost:
            best_cost, best_candidate_center_set = cost, candidate_center_set
    return dataset[list(best_candidate_center_set)], best_cost # type: ignore

### Vectorized + parallelized version of the brute-force algo

In [9]:
import numpy as np
from itertools import combinations
from concurrent.futures import ProcessPoolExecutor
from scipy.spatial.distance import cdist

def _eval_chunk(combos_chunk):
    # D is inherited from the parent process
    best_cost, best_combo = np.inf, None
    for combo in combos_chunk:
        cost = D[:, list(combo)].min(axis=1).max()
        if cost < best_cost:
            best_cost, best_combo = cost, combo
    return best_cost, best_combo

def k_center_brute_parallel(dataset, k, metric="euclidean",
                            n_workers=None, chunksize=4000):
    global D
    D = cdist(dataset, dataset, metric=metric)        # set before the pool forks
    combos = list(combinations(range(len(dataset)), k))
    chunks = [combos[i:i+chunksize] for i in range(0, len(combos), chunksize)]
    with ProcessPoolExecutor(max_workers=n_workers) as ex:   # n_workes = None defaults to os.cpu_count()
        best_cost, best_combo = min(ex.map(_eval_chunk, chunks), key=lambda r: r[0])
    return dataset[list(best_combo)], best_cost

#### Time comparison between the two versions

In [ ]:
import time 

dataset = build_dataset(n=500,dim=2)
start = time.time()
optimal_centers, optimal_cost= k_center_brute_vectorized(dataset=dataset, k = 3)
elapsed = time.time() - start
print(f"optimal_center_set: {optimal_centers} with cost: {optimal_cost}")
print(f"Time to compute optimal solution with vectorized brute force algo, for a dataset with {len(dataset)} points with {k} centers : {elapsed} seconds")

start = time.time()
optimal_centers, optimal_cost= k_center_brute_parallel(dataset=dataset, k = 3)
elapsed = time.time() - start
print(f"optimal_center_set: {optimal_centers} with cost: {optimal_cost}")
print(f"Time to compute optimal solution with vectorized brute force algo, for a dataset with {len(dataset)} with {k} centers : {elapsed} seconds")

optimal_center_set: [[ 28 119]
 [104  15]
 [132 126]] with cost: 93.1933474020544
Time to compute optimal solution with vectorized brute force algo, for a dataset with 500 points and 3 centers : 64.46469473838806 seconds
optimal_center_set: [[ 28 119]
 [104  15]
 [132 126]] with cost: 93.1933474020544
Time to compute optimal solution with vectorized brute force algo, for a dataset with 500 and 3 centers points : 10.171764135360718 seconds


# K-center greedy 2-approximation algorithm - gonzalez

Choose the center farthest away from the rest. Given a set of centers $C$, a new center $c$ is decided by solving: $$c = \argmax_p \min_{c\in C} d(p,c)$$ $c $ is the worst served point for the current set of centers. So the intution is that if $|C| \lt k$, pick the new center to be the worst served point for the current clustering, i.e the one with the highest cost. 

In [11]:
k = 3
print(f"number of possible different clusterings with {k} centers, in a dataset of {len(dataset)} points: {len(candidate_center_sets)}")

def k_center_gonzalez(dataset: np.ndarray, d, k: int):
    if len(dataset) < k:
        raise ValueError(f"Dataset must have at least {k} points")

    first_center_idx = np.random.randint(len(dataset))
    centers = [dataset[first_center_idx]]

    for _ in range(k - 1):  # first center already chosen
        point_distances = [min(d(p, c) for c in centers) for p in dataset]
        farthest_idx = max(range(len(dataset)), key=lambda i: point_distances[i])
        centers.append(dataset[farthest_idx])

    point_distances = [min(d(p, c) for c in centers) for p in dataset]
    farthest_idx = max(range(len(dataset)), key=lambda i: point_distances[i])
    cost = point_distances[farthest_idx]
    return centers, cost

dataset = build_dataset(n=500,dim=2)
centers, cost = k_center_gonzalez(dataset=dataset,d=euclidean_distance,k=k)
print(f"greedy centers: {centers} with corresponding cost: {cost}")




number of possible different clusterings with 3 centers, in a dataset of 500 points: 120
greedy centers: [array([148, 145]), array([5, 6]), array([  1, 183])] with corresponding cost: 148.7615541731129


# Brute-force algorithm vs Gonzalez algorithm cost comparison
Expected result is that the cost of the clustering given by the Gonzales algorithm should be at most twice as much as the brute-force algorithm. The expriment is conducted as follows: 

1. Run brute-force algorithm once and save the cost
2. Run Gonzalez's algorithm x times and save the costs
3. Take the mean of the costs from 2. then the ratio between that and the cost from 1.

In [16]:
np.random.seed(1)
dataset = build_dataset(n=500, max_val = 20000, dim=2) #n=500 OOM memory othewise LOL
np.random.seed()

def k_center_gonzalez_experiment(dataset: np.ndarray, iterations: int):
    k = 3
    start = time.time()
    _, optimal_cost = k_center_brute_parallel(dataset=dataset, k=k)
    elapsed = time.time() - start
    print(f"Time to compute optimal clustering on a dataset of size {len(dataset)} with {k} centers: {elapsed} seconds")

    gonzalez_costs = []
    start = time.time()
    for i in range(iterations):
        _, cost = k_center_gonzalez(dataset=dataset,d=euclidean_distance, k=k)
        gonzalez_costs.append(cost)
        approximation_ratio = cost / optimal_cost
        if approximation_ratio > 1.9:
            print(f"HIGH approx ratio at iteration {i}: {approximation_ratio}")
    elapsed = time.time() - start
    print(f"Time to compute {iterations} clusterings using Gonzalez's algorithm on a dataset of size {len(dataset)}: {elapsed} seconds ")
    
    gonzalez_costs_mean = np.mean(gonzalez_costs)
    approximation_ratio = gonzalez_costs_mean / optimal_cost
    return approximation_ratio

iterations = 1000
approximation_ratio = k_center_gonzalez_experiment(dataset=dataset, iterations = iterations) 
print(f" Expected approximation ratio is at most 2. Actual ratio after {iterations} iterations: {approximation_ratio}.")


Time to compute optimal clustering on a dataset of size 500 with 3 centers: 10.42641806602478 seconds
HIGH approx ratio at iteration 65: 1.941411605786777
HIGH approx ratio at iteration 151: 1.902073427941815
HIGH approx ratio at iteration 309: 1.90059599821186
HIGH approx ratio at iteration 336: 1.941411605786777
HIGH approx ratio at iteration 483: 1.941411605786777
HIGH approx ratio at iteration 721: 1.902073427941815
HIGH approx ratio at iteration 924: 1.902073427941815
HIGH approx ratio at iteration 957: 1.941411605786777
HIGH approx ratio at iteration 971: 1.941411605786777
HIGH approx ratio at iteration 976: 1.9232464187029983
HIGH approx ratio at iteration 997: 1.902073427941815
Time to compute 1000 clusterings using Gonzalez's algorithm on a dataset of size 500: 6.541710138320923 seconds 
 Expected approximation ratio is at most 2. Actual ratio after 1000 iterations: 1.4228753439340946.


In [271]:
a = np.matrix('1 2; 3 4')
dataset = build_dataset(10)
candidate_center_sets = list(combinations(dataset,3))
print(candidate_center_sets)
print(list(candidate_center_sets[0]))
a[:,0]


[(array([8]), array([32]), array([43])), (array([8]), array([32]), array([49])), (array([8]), array([32]), array([102])), (array([8]), array([32]), array([103])), (array([8]), array([32]), array([129])), (array([8]), array([32]), array([156])), (array([8]), array([32]), array([172])), (array([8]), array([32]), array([193])), (array([8]), array([43]), array([49])), (array([8]), array([43]), array([102])), (array([8]), array([43]), array([103])), (array([8]), array([43]), array([129])), (array([8]), array([43]), array([156])), (array([8]), array([43]), array([172])), (array([8]), array([43]), array([193])), (array([8]), array([49]), array([102])), (array([8]), array([49]), array([103])), (array([8]), array([49]), array([129])), (array([8]), array([49]), array([156])), (array([8]), array([49]), array([172])), (array([8]), array([49]), array([193])), (array([8]), array([102]), array([103])), (array([8]), array([102]), array([129])), (array([8]), array([102]), array([156])), (array([8]), ar

matrix([[1],
        [3]])